# Step 1: Prepare features

Ce notebook prépare les attributs au cas par cas:
- traite la connectivité
- complète l'attribut vitesse OSM
- merge les fichiers de source différente: VD, FR, GE pour obtenir 1 fichier GG

In [1]:
import importlib
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point, box, LineString
import os
import folium
from folium import Choropleth, CircleMarker, GeoJson
import branca.colormap as cm
from IPython.display import display
pd.set_option('display.max_columns', None)

import network_connectivity
import network_slope
importlib.reload(network_connectivity)
importlib.reload(network_slope)

from network_connectivity import *
from network_slope import *



## Imports

#### Import des segments

In [2]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters")
territory = 'GG' # GG or GE or OTC (need adaptation in attribute table, for later)
network = "walk"  # walk or bike


input_file_path = '../../Data/input'
output_step1_path=f"../../Data/output/{network}/{territory}/step-1"
output_step2_path=f"../../Data/output/{network}/{territory}/step-2"
output_step3_path=f"../../Data/output/{network}/{territory}/step-3"
save_path = f"../../Data/output/{network}/{territory}/step-1"
include = f"include_in_index_{territory}"

save_filtered_attributes = True

# Load segments GeoDataFrame (with 'segment_id')
print("Loading segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
segmented_net = segmented_net.to_crs(operation_crs)

# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):
    # Clean geometries first
    gdf["geometry"] = gdf["geometry"].apply(lambda geom: geom.buffer(0) if geom is not None and not geom.is_valid else geom)
    print("Geometries cleaned")

    if not save_filtered_attributes:
        print("Note : Save option is disabled.")
        return

    # Parse save formats: comma-separated list allowed
    raw = str(row.get('save_format', '') or '')
    formats = [f.strip().lower() for f in raw.split(',') if f.strip()]
    if not formats:
        formats = ['csv']  # default fallback

    for fmt in formats:
        try:
            if fmt == 'parquet':
                dirpath = f'{output_step2_path}/parquet_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_parquet(f"{dirpath}/{attribute}.parquet")
                print(f"Filtered data saved for attribute: {attribute} in format: parquet")

            elif fmt == 'gpkg' or fmt == 'geopackage':
                dirpath = f'{output_step2_path}/gpkg_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_file(f"{dirpath}/{attribute}.gpkg", driver="GPKG")
                print(f"Filtered data saved for attribute: {attribute} in format: gpkg")

            elif fmt == 'csv':
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                # to_csv may not preserve geometry consistently; keep original behavior
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Filtered data saved for attribute: {attribute} in format: csv")

            else:
                # Unknown format -> fallback to csv and warn
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Warning: Unknown save format '{fmt}' for attribute {attribute}. Data saved as csv.")
        except Exception as e:
            print(f"Error saving {attribute} as {fmt}: {e}")

Set parameters
Loading segments...


In [3]:
# Pour OTC : créer colonne infra_bike depuis TYPE_AMENA (équivalent de la classification OSM Step 0)
if territory == 'OTC':
    TYPE_AMENA_TO_INFRA_BIKE = {
        'Piste':          'piste_cyclable',
        'Bande':          'bande_cyclable',
        'Voie Bus':       'voie_bus',
        'Dérogation 2R':  'sur_chaussee',
        'Sas':            'sas_velo',
        'Contresens':     'contresens',
    }
    if 'TYPE_AMENA' in segmented_net.columns:
        segmented_net['infra_bike'] = segmented_net['TYPE_AMENA'].map(TYPE_AMENA_TO_INFRA_BIKE).fillna('inconnu')
    else:
        segmented_net['infra_bike'] = 'inconnu'
    print("OTC — infra_bike créé depuis TYPE_AMENA :")
    print(segmented_net['infra_bike'].value_counts())

#### Import des attributs

In [4]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info_{network}_urbain.xlsx", sheet_name="attributs_info")
attributs_info = attributs_info.loc[~attributs_info[include].isin(["False", "Unavailable"])].copy()  # Filter out rows where include_in_index is False or 'Unavailable'

attributs_info

,Class,meta_attribute,attribute,include_in_index_GG,include_in_index_GE,source_type_GE,source_path_GE,file_name_GE,source_type_VD,source_path_VD,file_name_VD,source_type_FR,source_path_FR,file_name_FR,source_type_GG,source_path_GG,file_name_GG,merge_rule_GG,initial_weight,class_weight,buffer_size_GG,buffer_size_GE,impact_attribut,geometry_type_GE,how_GE,value_column_GE,geometry_type_GG,how_GG,value_column_GG,clip_GG,clip_GE,missing_value_strategy,filter_column,filter_values,crs,save_format,Commentaire
0,Attractivité,lac_cours_deau,lac_cours_deau,True,True,sitg,attributs/GE/lac_cours_deau,CAD_NATURE_SOL-SHP/CAD_NATURE_SOL.shp,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.50,50,50,favorable,polygon,presence,NaN,polygon,presence,NaN,NaN,NaN,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",ok
1,Attractivité,fontaine,fontaine,True,True,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.50,50,50,favorable,point,count,NaN,point,count,NaN,2.00,2.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",ok
2,Attractivité,espaces_ouverts,espaces_ouverts,True,True,sitg,attributs/GE/espaces_ouverts,OBS_EQUIPEMENTS_ESPACES_PUB-SHP/OBS_EQUIPEMENT...,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.50,1,1,favorable,polygon,presence,NaN,polygon,presence,NaN,NaN,2.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB
3,Attractivité,amenite,rez_actif,True,True,sitg,attributs/GE/rez_actif,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.50,50,50,favorable,point,count,NaN,point,count,NaN,4.00,10.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB
4,Attractivité,transport_public,tp,True,True,sitg,attributs/GE/tp,TPG_ARRETS-SHP/TPG_ARRETS.shp,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.99,150,150,favorable,point,presence,NaN,point,presence,NaN,NaN,NaN,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB
5,Attractivité,amenite,amenite,True,True,sitg,attributs/GE/amenite,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.50,75,75,favorable,point,count,NaN,point,count,NaN,2.00,6.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB
7,Commodité,bruit,bruit,True,True,sitg,attributs/GE/bruit,SPBR_SECTEUR_EXPOSE_AU_BRUIT_2025/SPBR_SECTEUR...,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/bruit,bruit_classes_grand_geneve.tif,NaN,0.5,0.50,100,1,defavorable,polygon,presence,NaN,point,raster,bruit_norm,NaN,NaN,mean_if_missing,filtered,1,2056,"parquet, csv, gpkg",NaN
8,Commodité,temperature,temperature,True,True,sitg,attributs/GE/temperature,CLIMAT_TEMPERATURE_14H00_P1_2020/CLIMAT_TEMPER...,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/temperature,LST_GrandGeneve_2023.tif,NaN,0.5,0.50,100,250,defavorable,point,raster,temperature,point,raster,temperature,40.00,40.0,keep_nan,filtered,1,2056,"parquet, csv, gpkg",TB
9,Commodité,conflit_md,conflit_md,True,True,sitg,attributs/GE/network_couche_OCT.shp,RP_final_25112025.shp,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.50,1,1,defavorable,line,mean,Partage_us_score,point,sum,severity_score,72.00,2.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB
10,Commodité,environnement_ouvert_naturel,environnement_ouvert_naturel,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sitg,attributs/GG/environnement_ouvert_naturel,SIPV_MN_CARTO_GG.gpkg,NaN,0.0,0.00,30,30,favorable,polygon,area_ratio,NaN,polygon,area_ratio,NaN,0.99,NaN,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",NaN


## Connectivité du réseau

In [5]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'connectivite'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"Processing attribute: {attribute}")

# Ensure geometry column is set to avoid spatial index errors
segmented_net = segmented_net.set_geometry("geometry")
print(segmented_net.columns)

# 1. Ajouter u, v, key si besoin
segmented_net = add_uv_columns(segmented_net)

# 2. Calculer les métriques et le score d’alternatives
segmented_net_index = compute_connectivity_metrics(
    segmented_net,
    buffer_m=35,
    compute_betweenness=False,
    betweenness_k=None,
    crs_meter_epsg=2056,
    main_metrics_only=False,
    conn_index_metric="conn_branching_in_buffer",
)

segmented_net_index['filtered'] = 1


# Preview
segmented_net_index.to_crs(target_crs).head()

Data initialized
Processing attribute: connectivite
Index(['u', 'v', 'key', 'walk_class', 'source', 'highway', 'osm_id', 'id',
       'element', 'footway', 'sidewalk', 'name', 'maxspeed', 'service',
       'access', 'foot', 'tracktype', 'surface', 'smoothness', 'len_m',
       'walk_role', 'proxy_dedicated_covered_len_m',
       'proxy_dedicated_coverage_ratio', 'edge_id', 'adj_osm_id',
       'adj_highway', 'adj_speed_kmh', 'adj_dist_m', 'speed_ctx_kmh', 'length',
       'geometry', 'segment_id'],
      dtype='object')


,u,v,key,walk_class,source,highway,osm_id,id,element,footway,sidewalk,name,maxspeed,service,access,foot,tracktype,surface,smoothness,len_m,walk_role,proxy_dedicated_covered_len_m,proxy_dedicated_coverage_ratio,edge_id,adj_osm_id,adj_highway,adj_speed_kmh,adj_dist_m,speed_ctx_kmh,length,geometry,segment_id,length_m,conn_deadend_flag,conn_intersection_flag,conn_nodes_in_buffer,conn_intersections_in_buffer,conn_branching_in_buffer,conn_index_score,filtered
0,-3671742797194614437,-9140018067926386715,000000,walk_dedicated,OSM_highway_dedicated,footway,way/4272878,4272878.0,way,None,None,None,None,None,None,None,None,concrete,None,5.920195,dedicated,NaN,NaN,0,None,None,NaN,NaN,NaN,5.920195,"LINESTRING (6.1215 46.20185, 6.12157 46.20187)",000000,5.920195,False,True,4,1,0.114063,0.114063,1
1,-5781459007026747408,8604236943511184175,000001,walk_track,OSM_highway_dedicated,track,way/4297439,4297439.0,way,None,None,Chemin du Milieu,None,None,None,None,grade2,None,None,701.230056,dedicated,NaN,NaN,1,None,None,NaN,NaN,NaN,50.000000,"LINESTRING (6.08229 46.30288, 6.08185 46.3032)",000001,50.000000,False,True,3,1,0.114063,0.114063,1
2,8604236943511184175,-1121181787614706977,000002,walk_track,OSM_highway_dedicated,track,way/4297439,4297439.0,way,None,None,Chemin du Milieu,None,None,None,None,grade2,None,None,701.230056,dedicated,NaN,NaN,1,None,None,NaN,NaN,NaN,50.000000,"LINESTRING (6.08185 46.3032, 6.0814 46.30353)",000002,50.000000,False,False,2,0,0.100000,0.100000,1
3,-1121181787614706977,7016047240835109748,000003,walk_track,OSM_highway_dedicated,track,way/4297439,4297439.0,way,None,None,Chemin du Milieu,None,None,None,None,grade2,None,None,701.230056,dedicated,NaN,NaN,1,None,None,NaN,NaN,NaN,50.000000,"LINESTRING (6.0814 46.30353, 6.08095 46.30385)",000003,50.000000,False,False,2,0,0.100000,0.100000,1
4,7016047240835109748,-7152654980191551096,000004,walk_track,OSM_highway_dedicated,track,way/4297439,4297439.0,way,None,None,Chemin du Milieu,None,None,None,None,grade2,None,None,701.230056,dedicated,NaN,NaN,1,None,None,NaN,NaN,NaN,50.000000,"LINESTRING (6.08095 46.30385, 6.08051 46.30418)",000004,50.000000,False,False,2,0,0.100000,0.100000,1


In [6]:
save(save_filtered_attributes, row, segmented_net_index.to_crs(target_crs)[["segment_id","geometry","conn_branching_in_buffer","conn_index_score","filtered"]], attribute)

Geometries cleaned
Filtered data saved for attribute: connectivite in format: parquet
Filtered data saved for attribute: connectivite in format: csv
Filtered data saved for attribute: connectivite in format: gpkg


In [7]:
segmented_net_index.to_crs(target_crs)[["key","segment_id","geometry","conn_branching_in_buffer","conn_index_score","filtered"]]

,key,segment_id,geometry,conn_branching_in_buffer,conn_index_score,filtered
0,000000,000000,"LINESTRING (6.1215 46.20185, 6.12157 46.20187)",0.114063,0.114063,1
1,000001,000001,"LINESTRING (6.08229 46.30288, 6.08185 46.3032)",0.114063,0.114063,1
2,000002,000002,"LINESTRING (6.08185 46.3032, 6.0814 46.30353)",0.100000,0.100000,1
3,000003,000003,"LINESTRING (6.0814 46.30353, 6.08095 46.30385)",0.100000,0.100000,1
4,000004,000004,"LINESTRING (6.08095 46.30385, 6.08051 46.30418)",0.100000,0.100000,1
...,...,...,...,...,...,...
555370,555370,555370,"LINESTRING (6.12008 46.22094, 6.12017 46.22088)",0.000000,0.156250,1
555371,555371,555371,"LINESTRING (6.12017 46.22088, 6.12041 46.22071...",0.212500,0.212500,1
555372,555372,555372,"LINESTRING (6.12003 46.22091, 6.12009 46.22087)",0.000000,0.170313,1
555373,555373,555373,"LINESTRING (6.12009 46.22087, 6.12036 46.22068...",0.212500,0.212500,1


## Pente

In [8]:
# Initialize
attribute = None
row = None
segmented_net_slope = None
print("Data initialized")

# Load
###----change here-----
attribute = 'pente'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"Processing attribute: {attribute}")

source_path_col = f"source_path_{territory}"
file_name_col = f"file_name_{territory}"
raster_path = f"{input_file_path}/{row[source_path_col]}/{row[file_name_col]}"

segmented_net_slope = compute_segment_slope(
    segmented_net,
    raster_path,
    crs_meter_epsg=2056,
    length_col="length",
    smooth_sigma=1.0,
    smooth_hops=2,
)
segmented_net_slope['pente'] = segmented_net_slope['slope_pct_abs_smoothed']
segmented_net_slope['pente_brute'] = segmented_net_slope['slope_pct_abs']
segmented_net_slope['filtered'] = segmented_net_slope['pente'].notna().astype(int)

print("\nDescriptive statistics (pente lissée):")
print(segmented_net_slope['pente'].describe())

segmented_net_slope.to_crs(target_crs)[[
    'segment_id',
    'geometry',
    'pente',
    'pente_brute',
    'elev_start_m',
    'elev_end_m',
    'elev_delta_m',
    'slope_pct_signed',
    'filtered',
]].head()

Data initialized
Processing attribute: pente

Descriptive statistics (pente lissée):
count    555375.000000
mean          5.650841
std           6.146237
min           0.000000
25%           2.263555
50%           4.041330
75%           7.105228
max         381.751411
Name: pente, dtype: float64


,segment_id,geometry,pente,pente_brute,elev_start_m,elev_end_m,elev_delta_m,slope_pct_signed,filtered
0,000000,"LINESTRING (6.1215 46.20185, 6.12157 46.20187)",2.057334,0.000000,14.854813,14.854813,0.000000,0.000000,1
1,000001,"LINESTRING (6.08229 46.30288, 6.08185 46.3032)",0.602740,0.081681,2.104920,2.064079,-0.040841,-0.081681,1
2,000002,"LINESTRING (6.08185 46.3032, 6.0814 46.30353)",1.709063,1.822778,2.064079,2.975468,0.911389,1.822778,1
3,000003,"LINESTRING (6.0814 46.30353, 6.08095 46.30385)",2.371369,4.104128,2.975468,0.923404,-2.052064,-4.104128,1
4,000004,"LINESTRING (6.08095 46.30385, 6.08051 46.30418)",1.842500,0.764747,0.923404,1.305777,0.382373,0.764747,1


In [9]:
save(
    save_filtered_attributes,
    row,
    segmented_net_slope.to_crs(target_crs)[[
        'segment_id',
        'geometry',
        'pente',
        'pente_brute',
        'elev_start_m',
        'elev_end_m',
        'elev_delta_m',
        'slope_pct_signed',
        'slope_pct_abs',
        'slope_pct_abs_smoothed',
        'filtered',
    ]],
    attribute,
 )

Geometries cleaned
Filtered data saved for attribute: pente in format: parquet
Filtered data saved for attribute: pente in format: csv
Filtered data saved for attribute: pente in format: gpkg


## Vitesse

In [ ]:

import re
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

# --- Paramètres --------------------------------------------------------------
edges_path = f"{input_file_path}/networkGG/walk_edges_graph.geojson"
tache_urbaine_path = f"{input_file_path}/network_agreg/AGGLO_TACHE_URBAINE_2022.gpkg"
out_dir = Path(f"{input_file_path}/attributs/GG/vitesse")
basename = "vitesse_all_edges"
operation_crs = "EPSG:2056"

# ============================================================================
# 1_1 · Vitesse par tronçon
#   1) base : tache urbaine 2022 dilatée de TACHE_BUFFER_M
#      -> 50 km/h (bâti) / 80 km/h (non bâti)
#   2) type de route (seulement sans maxspeed explicite) :
#        pedestrian, path, track, steps, corridor -> 1 km/h (pas de voiture)
#        living_street -> 20 km/h, residential -> 30 km/h, service -> 30 km/h, track -> 10 km/h
#   3) maxspeed OSM explicite et numérique -> écrase tout le reste
#   speed_kph = vitesse retenue ; vitesse_type = zone | osm_tag | type_inferred
# ============================================================================

TACHE_BUFFER_M = 200.0   # dilatation de la tache urbaine (0 = aucune)

SPEED_URBAN_KMH = 50.0
SPEED_RURAL_KMH = 80.0



MAXSPEED_COLS = ["maxspeed:forward", "maxspeed:backward", "maxspeed"]

# --- Chargement --------------------------------------------------------------
edges = gpd.read_file(edges_path)

tache_urbaine = gpd.read_file(tache_urbaine_path)
tache_urbaine["geometry"] = tache_urbaine.geometry.force_2d().make_valid()
tache_urbaine = tache_urbaine.to_crs(operation_crs)[["geometry"]]
if TACHE_BUFFER_M > 0:
    tache_urbaine["geometry"] = tache_urbaine.geometry.buffer(TACHE_BUFFER_M)


# --- Helpers -----------------------------------------------------------------
_NUMERIC_RE = re.compile(r"\d+(\.\d+)?")
_LISTLIKE = (list, tuple, set, np.ndarray)

def parse_explicit_maxspeed(v):
    """Nombre seul (ex. '50', 50) -> float. Tout le reste ('walk', 'CH:urban',
    '50 mph', '30;50', 'none'...) -> NaN."""
    if v is None or isinstance(v, _LISTLIKE):
        return np.nan
    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v) if np.isfinite(v) and v > 0 else np.nan
    s = str(v).strip()
    if _NUMERIC_RE.fullmatch(s):
        x = float(s)
        return x if x > 0 else np.nan
    return np.nan


def normalize_highway(v):
    """Liste OSM (ex. ['tertiary', 'primary']) -> première valeur."""
    if isinstance(v, _LISTLIKE):
        v = list(v)
        return str(v[0]) if len(v) else np.nan
    if v is None or pd.isna(v):
        return np.nan
    return str(v)


NO_CAR_OWN_HW = {"pedestrian", "corridor"}   # l'arête est elle-même un espace sans voiture
NO_CAR_SPEED_KMH = 0.0                        # repéré par vitesse_type = "no_car"
TYPE_SPEED_KMH = {"living_street": 20.0, "residential": 30.0, "service": 30.0}
TRACK_SPEED_KMH = 10.0                        # track isolé : véhicules agricoles


def compute_speed(edges_gdf, tache_gdf):
    df = edges_gdf.copy()
    if "speed_ctx_kmh" not in df.columns:
        raise ValueError("speed_ctx_kmh absent : relancer 0_1 (cellule vitesse de la section 7).")

    # 1) Base : milieu du tronçon dans la tache urbaine (inchangé)
    mid = gpd.GeoDataFrame(
        geometry=df.to_crs(operation_crs).geometry.interpolate(0.5, normalized=True),
        index=df.index, crs=operation_crs,
    )
    in_tache = gpd.sjoin(mid, tache_gdf, how="inner", predicate="intersects").index.unique()
    df["in_tache_urbaine"] = df.index.isin(in_tache)

    speed = pd.Series(np.where(df["in_tache_urbaine"], SPEED_URBAN_KMH, SPEED_RURAL_KMH),
                      index=df.index, dtype="float64")
    vtype = pd.Series("zone", index=df.index, dtype="object")

    # Route de référence : l'arête elle-même si c'est un axe routier gardé,
    # sinon la chaussée adjacente trouvée dans 0_1 (<= 15 m)
    hw = df["highway"].apply(normalize_highway)
    adj_hw = df["adj_highway"].apply(normalize_highway)
    is_proxy = df["walk_class"].eq("sidewalk_proxy_road_axis")
    has_adj = df["adj_dist_m"].notna()
    ref_hw = hw.where(is_proxy, adj_hw)

    # 2) Type de la route de référence
    type_speed = ref_hw.map(TYPE_SPEED_KMH)
    m = type_speed.notna()
    speed[m] = type_speed[m]
    vtype[m] = "type_inferred"

    # 3) Vitesse explicite : tag propre (proxy) ou chaussée adjacente (trottoir, traversée)
    ctx = pd.to_numeric(df["speed_ctx_kmh"], errors="coerce")
    m = ctx.notna() & (ctx > 0)
    speed[m] = ctx[m]
    vtype[m & is_proxy] = "osm_tag"
    vtype[m & ~is_proxy] = "osm_tag_adjacent"

    # 4) Pas de voiture : espace piéton en soi, ou aucune chaussée à côté
    isolated = ~is_proxy & ~has_adj
    m_track = isolated & hw.eq("track")
    speed[m_track] = TRACK_SPEED_KMH
    vtype[m_track] = "type_inferred"

    m_nocar = hw.isin(NO_CAR_OWN_HW) | (isolated & ~hw.eq("track"))
    speed[m_nocar] = NO_CAR_SPEED_KMH
    vtype[m_nocar] = "no_car"

    df["highway_norm"] = hw
    df["ref_highway"] = ref_hw
    df["speed_kph"] = speed
    df["vitesse_type"] = vtype
    return df


def print_speed_summary(df):
    length_m = df.to_crs(operation_crs).geometry.length
    summary = pd.DataFrame({
        "n_troncons": df.groupby("vitesse_type").size(),
        "longueur_km": length_m.groupby(df["vitesse_type"]).sum() / 1000,
    })
    summary["part_troncons_%"] = 100 * summary["n_troncons"] / summary["n_troncons"].sum()
    summary["part_longueur_%"] = 100 * summary["longueur_km"] / summary["longueur_km"].sum()
    print("Répartition par type de vitesse :")
    print(summary.round(1).to_string())

    z = df["vitesse_type"] == "zone"
    print(f"\nZone : {int((z & df['in_tache_urbaine']).sum())} tronçons à {SPEED_URBAN_KMH:g} km/h, "
          f"{int((z & ~df['in_tache_urbaine']).sum())} à {SPEED_RURAL_KMH:g} km/h")

    # Contrôle : classes de route qui reçoivent la vitesse de zone
    print("\nClasses highway notées par la zone (top 10) :")
    print(df.loc[z, "highway_norm"].value_counts(dropna=False).head(10).to_string())


def export_speed_layer(df, out_dir, basename):
    out_dir.mkdir(parents=True, exist_ok=True)
    keep = [c for c in ["edge_id", "u", "v", "key", "osm_id", "id", "element", "name",
                        "highway", "highway_norm", "walk_class", "ref_highway",
                        "adj_osm_id", "adj_highway", "adj_dist_m", "speed_ctx_kmh",
                        "in_tache_urbaine", "speed_kph", "vitesse_type",
                        "length", "len_m", "geometry"] if c in df.columns]
    out = df[keep].copy()
    # listes OSM éventuelles -> texte, pour l'export
    for c in out.columns.drop("geometry"):
        if out[c].map(lambda x: isinstance(x, _LISTLIKE)).any():
            out[c] = out[c].map(lambda x: ";".join(map(str, x)) if isinstance(x, _LISTLIKE) else x)

    out.to_file(out_dir / f"{basename}.geojson", driver="GeoJSON")
    out.to_parquet(out_dir / f"{basename}.parquet", index=False)
    print("\nExported:", out_dir / f"{basename}.geojson")
    print("Exported:", out_dir / f"{basename}.parquet")
    print("Missing speed_kph:", int(out["speed_kph"].isna().sum()))
    return out


# --- Exécution ---------------------------------------------------------------
speed_edges = compute_speed(edges, tache_urbaine)
print_speed_summary(speed_edges)
speed_gdf = export_speed_layer(speed_edges, out_dir, basename)

/Users/maximiliantrique/Documents/GitHub/projet_pieton_gg/venv/lib/python3.11/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(


Répartition par type de vitesse :
                  n_troncons  longueur_km  part_troncons_%  part_longueur_%
vitesse_type                                                               
no_car                 37257       3297.7             11.0             19.1
osm_tag                39770       1543.1             11.7              9.0
osm_tag_adjacent       43216       1078.5             12.7              6.3
type_inferred         192870       9337.9             56.8             54.2
zone                   26169       1981.9              7.7             11.5

Zone : 23934 tronçons à 50 km/h, 2235 à 80 km/h

Classes highway notées par la zone (top 10) :
highway_norm
unclassified    13803
tertiary         4210
track            3195
footway          2389
NaN              1485
path              902
steps             101
platform           84

Exported: ../../Data/input/attributs/GG/vitesse/vitesse_all_edges.geojson
Exported: ../../Data/input/attributs/GG/vitesse/vitesse_all_edges.parquet


## Traversées

Score par segment : `traversees = 1 − besoin × (1 − offre)`.
- **besoin** : selon la classe de vitesse du segment (`vitesse_all_edges.parquet`, jointure sur `edge_id`) : `no_car` ]−∞,10] · `slow_cars` ]10,30] · `normal` ]30,50] · `intense` ]50,+∞[ km/h.
- **offre** : traversées OSM (`walk_crossing_nodes.parquet`) dans un rectangle autour du segment (demi-largeur `CROSSING_BUFFER_M`, prolongé de `CROSSING_EXTEND_M` à chaque bout). `slow_cars` : 1 dès qu'il y a une traversée. `normal` / `intense` : 1 avec feux, 0.5 sans feux, 0 sans traversée.

La logique du score est dans `compute_crossing_score()` (bloc `>>> BLOC SCORE TRAVERSÉES`). À lancer après la cellule Vitesse.

In [9]:
# ============================================================================
# 1_1 · Traversées par segment
#   1) segments step1 + vitesse (vitesse_all_edges.parquet, jointure sur edge_id)
#   2) classe de vitesse : no_car ]-inf,10] · slow_cars ]10,30] · normal ]30,50] · intense ]50,+inf[
#   3) traversées OSM (walk_crossing_nodes.parquet) filtrées sur le tag crossing,
#      affectées à un segment si elles tombent dans un rectangle autour du segment
#      (demi-largeur CROSSING_BUFFER_M, segment prolongé de CROSSING_EXTEND_M à chaque bout)
#   4) score = 1 - besoin × (1 - offre)   -> voir compute_crossing_score()
# ============================================================================
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely

# --- Paramètres (à tuner) ----------------------------------------------------
attribute = "traversees"

crossings_path = f"{input_file_path}/networkGG/walk_crossing_nodes.parquet"
speed_path     = f"{input_file_path}/attributs/GG/vitesse/vitesse_all_edges.parquet"

CROSSING_BUFFER_M = 10.0   # gonflement en épaisseur, de CHAQUE côté du segment (largeur totale = 2×)
CROSSING_EXTEND_M = 5.0    # prolongement du segment à chaque extrémité, dans l'axe du segment

# Valeurs du tag `crossing` retenues. None = nœud highway=crossing sans tag crossing.
CROSSING_VALUES_KEPT = ["traffic_signals", "uncontrolled", "marked", "zebra", "unmarked", None]

# Une traversée est « à feux » si l'une de ces colonnes prend l'une de ces valeurs
# (les colonnes absentes du fichier sont ignorées)
SIGNAL_RULES = {
    "crossing": ["traffic_signals"],
}

# Classes de vitesse : bornes droites incluses
SPEED_BINS   = [-np.inf, 10, 30, 50, np.inf]
SPEED_LABELS = ["no_car", "slow_cars", "normal", "intense"]

# Besoin de traversées par classe (0 = aucun besoin, 1 = besoin maximal)
BESOIN_BY_CLASS = {"no_car": 0.0, "slow_cars": 0.3, "normal": 0.7, "intense": 1.0}

EXPORT_DEBUG_RECTANGLES = False   # True : exporte les rectangles en gpkg pour contrôle dans QGIS


# --- Helpers -----------------------------------------------------------------
_LISTLIKE = (list, tuple, set, np.ndarray)

def _first(v):
    """Valeur OSM éventuellement en liste -> première valeur ; NaN/None -> None."""
    if isinstance(v, _LISTLIKE):
        v = list(v)
        v = v[0] if len(v) else None
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    return str(v)


def load_crossings(path, crs):
    """Charge les nœuds de traversée (GeoParquet, ou parquet avec colonnes x/y ou lon/lat)."""
    try:
        gdf = gpd.read_parquet(path)
    except Exception:
        df = pd.read_parquet(path)
        for xc, yc, src_crs in [("x", "y", crs), ("lon", "lat", "EPSG:4326"), ("longitude", "latitude", "EPSG:4326")]:
            if {xc, yc} <= set(df.columns):
                gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[xc], df[yc]), crs=src_crs)
                break
        else:
            raise ValueError(f"Pas de géométrie reconnue dans {path} (colonnes : {list(df.columns)})")
    return gdf.to_crs(crs)


def filter_crossings(crossings, values_kept, signal_rules):
    """Garde les traversées dont le tag crossing est dans values_kept, et marque celles à feux."""
    cr = crossings.copy()
    tag = cr["crossing"].map(_first) if "crossing" in cr.columns else pd.Series(None, index=cr.index)
    keep_none = any(v is None for v in values_kept)
    keep_vals = {v for v in values_kept if v is not None}
    mask = tag.isin(keep_vals) | (tag.isna() & keep_none)

    print("Valeurs du tag crossing (avant filtre) :")
    print(tag.value_counts(dropna=False).to_string())

    is_signal = pd.Series(False, index=cr.index)
    for col, vals in signal_rules.items():
        if col in cr.columns:
            is_signal |= cr[col].map(_first).isin(vals)
    cr["is_signal"] = is_signal

    cr = cr.loc[mask, ["is_signal", "geometry"]]
    print(f"\nTraversées retenues : {len(cr):,} / {len(crossings):,} "
          f"(dont à feux : {int(cr['is_signal'].sum()):,})")
    return cr


def segment_rectangles(lines, half_width, extend):
    """Rectangle autour de chaque segment : ligne prolongée de `extend` à chaque bout,
    puis buffer à bouts plats de `half_width`. Vectorisé (shapely 2)."""
    geoms = np.asarray(lines.geometry.values)
    is_line = shapely.get_type_id(geoms) == 1           # LineString
    out = geoms.copy()

    if extend > 0 and is_line.any():
        lg = geoms[is_line]
        coords, idx = shapely.get_coordinates(lg, return_index=True)
        coords = coords.copy()
        starts = np.r_[0, np.flatnonzero(np.diff(idx)) + 1]
        ends = np.r_[starts[1:] - 1, len(idx) - 1]
        ok = ends > starts                              # au moins 2 points

        s, e = starts[ok], ends[ok]
        for p, q in [(s, s + 1), (e, e - 1)]:            # p = extrémité, q = point voisin
            d = coords[p] - coords[q]
            n = np.linalg.norm(d, axis=1, keepdims=True)
            unit = np.divide(d, n, out=np.zeros_like(d), where=n > 0)
            coords[p] = coords[p] + unit * extend
        out[is_line] = shapely.linestrings(coords, indices=idx)

    rect = shapely.buffer(out, half_width, cap_style="flat")
    return gpd.GeoSeries(rect, index=lines.index, crs=lines.crs)


def count_crossings(segments, crossings, half_width, extend):
    """Nombre de traversées (et à feux) dans le rectangle de chaque segment."""
    rect = gpd.GeoDataFrame(geometry=segment_rectangles(segments, half_width, extend))
    hits = gpd.sjoin(crossings, rect, how="inner", predicate="within")
    n_all = hits.groupby("index_right").size()
    n_sig = hits.loc[hits["is_signal"]].groupby("index_right").size()
    res = pd.DataFrame(index=segments.index)
    res["n_traversees"] = n_all.reindex(res.index, fill_value=0).astype(int)
    res["n_traversees_feux"] = n_sig.reindex(res.index, fill_value=0).astype(int)
    return res, rect


# ============================================================================
# >>> BLOC SCORE TRAVERSÉES : c'est ici que se règle la logique du score
# ============================================================================
def compute_crossing_score(vitesse_classe, n_traversees, n_traversees_feux,
                           besoin_by_class=BESOIN_BY_CLASS):
    """
    score = 1 - besoin × (1 - offre)

    besoin : selon la classe de vitesse (besoin_by_class)
    offre  :
      - no_car, slow_cars : 1 dès qu'il y a une traversée, sinon 0 (pas de bonus feux)
      - normal, intense   : 1 si au moins une traversée à feux,
                            0.5 si traversée(s) sans feux uniquement,
                            0 si aucune traversée
    Classe inconnue (vitesse manquante) -> NaN.
    """
    cls = pd.Series(vitesse_classe).astype("object")
    n = pd.Series(n_traversees, index=cls.index)
    n_sig = pd.Series(n_traversees_feux, index=cls.index)

    besoin = cls.map(besoin_by_class).astype(float)

    has_any = n > 0
    has_sig = n_sig > 0
    car_heavy = cls.isin(["normal", "intense"])

    offre = np.where(
        car_heavy,
        np.where(has_sig, 1.0, np.where(has_any, 0.5, 0.0)),
        np.where(has_any, 1.0, 0.0),
    )
    offre = pd.Series(offre, index=cls.index).where(besoin.notna())

    score = 1 - besoin * (1 - offre)
    return besoin, offre, score
# ============================================================================
# <<< FIN BLOC SCORE TRAVERSÉES
# ============================================================================


# --- Exécution ---------------------------------------------------------------
# 1) Segments + vitesse (jointure sur edge_id)
seg = segmented_net.to_crs(operation_crs).copy()
speed = (pd.read_parquet(speed_path, columns=["edge_id", "speed_kph"])
           .drop_duplicates("edge_id"))
seg = seg.merge(speed, on="edge_id", how="left")
print(f"Segments sans vitesse : {int(seg['speed_kph'].isna().sum()):,} / {len(seg):,}")

seg["vitesse_classe"] = pd.cut(seg["speed_kph"], bins=SPEED_BINS, labels=SPEED_LABELS, right=True)
print("\nClasses de vitesse (segments) :")
print(seg["vitesse_classe"].value_counts(dropna=False).to_string())

# 2) Traversées
crossings = filter_crossings(load_crossings(crossings_path, operation_crs),
                             CROSSING_VALUES_KEPT, SIGNAL_RULES)
counts, rectangles = count_crossings(seg, crossings, CROSSING_BUFFER_M, CROSSING_EXTEND_M)
seg = seg.join(counts)

# 3) Score
seg["besoin_traversee"], seg["offre_traversee"], seg[attribute] = compute_crossing_score(
    seg["vitesse_classe"], seg["n_traversees"], seg["n_traversees_feux"])
seg["filtered"] = seg[attribute].notna().astype(int)

print(f"\nSegments avec ≥ 1 traversée : {int((seg['n_traversees'] > 0).sum()):,}")
print("\nScore moyen par classe :")
print(seg.groupby("vitesse_classe", observed=False)[[attribute, "offre_traversee"]].mean().round(3).to_string())

segmented_net_crossings = seg
segmented_net_crossings.to_crs(target_crs)[[
    "segment_id", "geometry", "speed_kph", "vitesse_classe",
    "n_traversees", "n_traversees_feux", "besoin_traversee", "offre_traversee", attribute, "filtered",
]].head()

Segments sans vitesse : 0 / 555,375

Classes de vitesse (segments) :
vitesse_classe
slow_cars    272228
no_car       171268
normal        95731
intense       16148
Valeurs du tag crossing (avant filtre) :
crossing
uncontrolled                    8168
None                            2621
marked                          2280
unmarked                        2079
traffic_signals                 1820
zebra                            185
informal                          11
pavement                           2
uncontrolled;traffic_signals       1
island                             1

Traversées retenues : 17,153 / 17,168 (dont à feux : 1,820)

Segments avec ≥ 1 traversée : 68,854

Score moyen par classe :
                traversees  offre_traversee
vitesse_classe                             
no_car               1.000            0.007
slow_cars            0.744            0.147
normal               0.413            0.162
intense              0.065            0.065


,segment_id,geometry,speed_kph,vitesse_classe,n_traversees,n_traversees_feux,besoin_traversee,offre_traversee,traversees,filtered
0,000000,"LINESTRING (6.1215 46.20185, 6.12157 46.20187)",0.0,no_car,0,0,0.0,0.0,1.0,1
1,000001,"LINESTRING (6.08229 46.30288, 6.08185 46.3032)",10.0,no_car,0,0,0.0,0.0,1.0,1
2,000002,"LINESTRING (6.08185 46.3032, 6.0814 46.30353)",10.0,no_car,0,0,0.0,0.0,1.0,1
3,000003,"LINESTRING (6.0814 46.30353, 6.08095 46.30385)",10.0,no_car,0,0,0.0,0.0,1.0,1
4,000004,"LINESTRING (6.08095 46.30385, 6.08051 46.30418)",10.0,no_car,0,0,0.0,0.0,1.0,1


In [10]:
# Sauvegarde traversées
_rows = attributs_info[attributs_info["attribute"] == attribute]
row = _rows.iloc[0] if len(_rows) else pd.Series({"save_format": "parquet,gpkg"})
if not len(_rows):
    print(f"'{attribute}' absent de attributs_info (ou exclu) : sauvegarde en parquet + gpkg par défaut")

out_cols = ["segment_id", "geometry", "speed_kph", "vitesse_classe",
            "n_traversees", "n_traversees_feux", "besoin_traversee", "offre_traversee",
            attribute, "filtered"]
to_save = segmented_net_crossings[out_cols].copy()
to_save["vitesse_classe"] = to_save["vitesse_classe"].astype("object")   # catégorie -> texte pour gpkg/csv
save(save_filtered_attributes, row, to_save.to_crs(target_crs), attribute)

if EXPORT_DEBUG_RECTANGLES:
    dbg_dir = f"{output_step2_path}/gpkg_attributs"
    os.makedirs(dbg_dir, exist_ok=True)
    rect_out = gpd.GeoDataFrame(segmented_net_crossings[["segment_id", "n_traversees", "n_traversees_feux"]],
                                geometry=rectangles.geometry.values, crs=operation_crs)
    rect_out.to_file(f"{dbg_dir}/{attribute}_rectangles.gpkg", driver="GPKG")
    crossings.to_file(f"{dbg_dir}/{attribute}_crossings_retenues.gpkg", driver="GPKG")
    print("Rectangles et traversées retenues exportés pour contrôle QGIS")

'traversees' absent de attributs_info (ou exclu) : sauvegarde en parquet + gpkg par défaut
Geometries cleaned
Filtered data saved for attribute: traversees in format: parquet
Filtered data saved for attribute: traversees in format: gpkg


## Merge files

In [11]:
#Helper

territories = ['GE', 'FR', 'VD']

# ── Chargement générique avec gestion des valeurs manquantes ──
def load_territory(attributs_info, attribute, territory, input_file_path, target_crs):
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    source_path = row.get(f'source_path_{territory}')
    file_name   = row.get(f'file_name_{territory}')

    # Cas manquants : cellule vide, NaN, ou chemin inexistant
    if pd.isna(source_path) or pd.isna(file_name):
        print(f"[{territory}] Pas de fichier défini: territoire ignoré.")
        return None

    full_path = f"{input_file_path}/{source_path}/{file_name}"

    if not os.path.exists(full_path):
        print(f"[{territory}] Fichier introuvable : {full_path} territoire ignoré.")
        return None

    gdf = gpd.read_file(full_path)
    return gdf.to_crs(target_crs)

### Sens inverse

In [12]:
# ── CAHNGE HERE ──────────────────────────────────────────────
attribute = 'sens_inverse'
#──────────────────────────────────────────────

# ── Chargement de tous les territoires ────────────────────────
gdfs = {}
for territory in territories:
    result = load_territory(
        attributs_info, attribute, territory, input_file_path, target_crs
    )
    if result is not None:
        result['territory'] = territory
        gdfs[territory] = result

# ── Merge uniquement si au moins un territoire chargé ─────────
if gdfs:
    attribute_merged = gpd.GeoDataFrame(
        pd.concat(gdfs.values(), ignore_index=True),
        crs=target_crs
    )
    print(f"Merge réussi : {len(attribute_merged)} lignes — territoires : {list(gdfs.keys())}")
else:
    print(f"Aucun fichier trouvé pour l'attribut '{attribute}'.")
    attribute_merged = None


#Save merged file into input/attributs/GG/filename

if attribute_merged is not None:
    output_dir  = os.path.join(input_file_path, 'attributs', 'GG', attribute)
    output_file = os.path.join(output_dir, f"{attribute}.gpkg")
    
    os.makedirs(output_dir, exist_ok=True)
    attribute_merged.to_file(output_file, driver='GPKG')
    
    print(f"Fichier sauvegardé : {output_file}")
else:
    print(f"Aucun fichier à sauvegarder pour '{attribute}'.")

IndexError: single positional indexer is out-of-bounds

### Tourner à droite ###

In [ ]:
# ── CHANGE HERE ──────────────────────────────────────────────
attribute = 'tourner_droite'
#──────────────────────────────────────────────

# ── Chargement de tous les territoires ────────────────────────
gdfs = {}
for territory in territories:
    result = load_territory(
        attributs_info, attribute, territory, input_file_path, target_crs
    )
    if result is not None:
        result['territory'] = territory
        gdfs[territory] = result

# ── Merge uniquement si au moins un territoire chargé ─────────
if gdfs:
    attribute_merged = gpd.GeoDataFrame(
        pd.concat(gdfs.values(), ignore_index=True),
        crs=target_crs
    )
    print(f"Merge réussi : {len(attribute_merged)} lignes — territoires : {list(gdfs.keys())}")
else:
    print(f"Aucun fichier trouvé pour l'attribut '{attribute}'.")
    attribute_merged = None


#Save merged file into input/attributs/GG/filename

if attribute_merged is not None:
    output_dir  = os.path.join(input_file_path, 'attributs', 'GG', attribute)
    output_file = os.path.join(output_dir, f"{attribute}.gpkg")
    
    os.makedirs(output_dir, exist_ok=True)
    attribute_merged.to_file(output_file, driver='GPKG')
    
    print(f"Fichier sauvegardé : {output_file}")
else:
    print(f"Aucun fichier à sauvegarder pour '{attribute}'.")